## importo las librerias necesarias

In [3]:
import pandas as pd
import matplotlib as mtp
import numpy as np


## descargo los files desde la carpeta descarga en mi C y compruebo si hay mas de un csv en el zip
## en caso haya mas que uno, cargo los csv en un diccionario DF de pandas

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

file_zip = Path.home() / "Desktop" / "downloads" / "archive International football results from 1872 to 2026.zip"

with zipfile.ZipFile(file_zip) as z:
    csv_files = [nombre for nombre in z.namelist() if nombre.lower().endswith(".csv")]
    dataframes = {
        Path(nombre).stem: pd.read_csv(
            z.open(nombre),
            encoding="cp1252",
            encoding_errors="replace"
        )
        for nombre in csv_files 
    }

print("CSV cargados:", list(dataframes.keys()))

CSV cargados: ['former_names', 'goalscorers', 'results', 'shootouts']


## Los cuatro CSV ya están cargados en dataframes
## veo una muestra de cada uno en el notebook

In [6]:
from IPython.display import display

for nombre, df in dataframes.items():
    print(f"\n--- {nombre} ({df.shape[0]} filas, {df.shape[1]} columnas) ---")
    display(df.head())


--- former_names (36 filas, 4 columnas) ---


,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,CuraÃ§ao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01



--- goalscorers (47914 filas, 8 columnas) ---


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,JosÃ© Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino GradÃ­n,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino GradÃ­n,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,JosÃ© Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False



--- results (49945 filas, 9 columnas) ---


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False



--- shootouts (700 filas, 5 columnas) ---


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1970-06-26,Hong Kong,Malaysia,Malaysia,NaN
2,1970-06-26,Indonesia,South Korea,South Korea,NaN
3,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
4,1972-05-07,South Korea,Iraq,Iraq,NaN


In [7]:
display(df.head(25))

,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1970-06-26,Hong Kong,Malaysia,Malaysia,NaN
2,1970-06-26,Indonesia,South Korea,South Korea,NaN
3,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
4,1972-05-07,South Korea,Iraq,Iraq,NaN
5,1972-05-17,Thailand,South Korea,South Korea,NaN
6,1972-05-19,Thailand,Cambodia,Thailand,NaN
7,1973-04-21,Senegal,Ghana,Ghana,NaN
8,1973-06-14,Guinea,Mali,Guinea,NaN
9,1973-06-14,Mauritius,Tanzania,Mauritius,NaN


## hago un info de los 4 df

In [8]:
for nombre, df in dataframes.items():
    print(f"\n--- {nombre} ---")
    df.info()


--- former_names ---
<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   current     36 non-null     str  
 1   former      36 non-null     str  
 2   start_date  36 non-null     str  
 3   end_date    36 non-null     str  
dtypes: str(4)
memory usage: 1.3 KB

--- goalscorers ---
<class 'pandas.DataFrame'>
RangeIndex: 47914 entries, 0 to 47913
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   date       47914 non-null  str    
 1   home_team  47914 non-null  str    
 2   away_team  47914 non-null  str    
 3   team       47914 non-null  str    
 4   scorer     47870 non-null  str    
 5   minute     47660 non-null  float64
 6   own_goal   47914 non-null  bool   
 7   penalty    47914 non-null  bool   
dtypes: bool(2), float64(1), str(5)
memory usage: 2.3 MB

--- results ---
<class 'pandas.DataFrame'

## no tiene sentido unificar los 4 df en un unico df porque los datos no son homogeneos y puede se que de muchos errores y datos duplicados
##  tengo que unificar los nombres de los paises para tener la cuenta de una sola entidad por pais - la actual - y evitar que se dispersen los datos

## Limpieza previa de los datos, empezando con las fechas que son todas type: str y que tengo que pasar a fecha real

In [9]:
# defino y renombro "former names" "results" "goalscorer" y "shootouts"; 
# los CSV estan cargados dentro del diccionario dataframes. 
# Uso sus claves para crear los alias y convertir las fechas

paises = dataframes["former_names"]
resultados = dataframes["results"]
marcadores = dataframes["goalscorers"]
penalties = dataframes["shootouts"]

#paso a fechas reales
paises["start_date"] = pd.to_datetime(paises["start_date"])
paises["end_date"] = pd.to_datetime(paises["end_date"])
resultados["date"] = pd.to_datetime(resultados["date"])
marcadores["date"] = pd.to_datetime(marcadores["date"])
penalties["date"] = pd.to_datetime(penalties["date"])

## unifico los nombres de los paises con el actual - me creo un diccionario de nombres

In [10]:
nombres_paises = dict(zip(
    dataframes["former_names"]["former"],
    dataframes["former_names"]["current"]
))


## remplazo los nombres de paises en los df resultados y marcadores

In [11]:
resultados["home_team"] = resultados["home_team"].replace(nombres_paises)
resultados["away_team"] = resultados["away_team"].replace(nombres_paises)

In [12]:
# compruebo si se han quedado nombres viejos que no se han sustituido

nombres_antiguos = paises["former"].tolist()

restantes_home = resultados["home_team"].isin(nombres_antiguos).sum()
restantes_away = resultados["away_team"].isin(nombres_antiguos).sum()

print(f"Nombres antiguos restantes en home_team: {restantes_home}")
print(f"Nombres antiguos restantes en away_team: {restantes_away}")

# si funciona sigo con los marcadores

Nombres antiguos restantes en home_team: 0
Nombres antiguos restantes en away_team: 0


In [13]:
marcadores["home_team"] = marcadores["home_team"].replace(nombres_paises)
marcadores["away_team"] = marcadores["away_team"].replace(nombres_paises)

## tengo que limpiar los NaN en el df marcadores, que es el unico que tiene NaN

In [14]:
print(marcadores.isnull().sum())

date           0
home_team      0
away_team      0
team           0
scorer        44
minute       254
own_goal       0
penalty        0
dtype: int64


In [15]:
# para los marcadores, sustituyo los NaN con "desconocido" para no perder valores

marcadores["scorer"] = marcadores["scorer"].fillna("desconocido")

print(marcadores.isnull().sum())

date           0
home_team      0
away_team      0
team           0
scorer         0
minute       254
own_goal       0
penalty        0
dtype: int64


In [16]:
# ¿que hago con los minutos? miro si hay una forma de renombrar los NaN para marcarlos y no perder datos
# puedo llenar los NaN de la misma forma de antes o marcarlos como minuto ej "120"

minutos_unicos = sorted(marcadores['minute'].unique())

print("Minutos únicos en el dataset:")
print(minutos_unicos)

Minutos únicos en el dataset:
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(30.0), np.float64(31.0), np.float64(32.0), np.float64(33.0), np.float64(34.0), np.float64(35.0), np.float64(36.0), np.float64(37.0), np.float64(38.0), np.float64(39.0), np.float64(40.0), np.float64(41.0), np.float64(42.0), np.float64(43.0), np.float64(44.0), np.float64(45.0), np.float64(46.0), np.float64(47.0), np.float64(48.0), np.float64(49.0), np.float64(50.0), np.float64(51.0), np.float64(52.0), np.float64(53.0), np.float64(54.0), np.flo

In [17]:
# no me compensa cambiar los NaN con un valor inventado, mejor indicar que falta el dato y marcar como desconocido

marcadores["minute"] = marcadores["minute"].fillna("desconocido")

print(marcadores.isnull().sum())

date         0
home_team    0
away_team    0
team         0
scorer       0
minute       0
own_goal     0
penalty      0
dtype: int64


## Ahora que tengo todos los df limpios, hago una copia para donde empezar a trabajar 

In [18]:
#indico la ruta del escritorio

escritorio = Path.home() / "Desktop"
nueva_carpeta = escritorio / "datos_futbol_limpios"

In [19]:
# creo la nueva carpeta

nueva_carpeta.mkdir(parents=True, exist_ok=True)
print(f"Carpeta creada en: {nueva_carpeta}")

Carpeta creada en: C:\Users\fabri\Desktop\datos_futbol_limpios


In [20]:
# y los df limpios

resultados.to_csv(nueva_carpeta / "resultados_limpio.csv", index=False)
marcadores.to_csv(nueva_carpeta / "marcadores_limpio.csv", index=False)
paises.to_csv(nueva_carpeta / "paises_limpio.csv", index=False)
penalties.to_csv(nueva_carpeta / "penalties_limpio.csv", index=False)